In [50]:
from google.colab import drive
import pickle
import pandas as pd
from collections import defaultdict
import random
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [51]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [52]:
# Copying 8k to Colabs local SSD
if not os.path.exists("/content/flickr8k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr8k.zip" /content/

In [53]:
if not os.path.isdir("/content/flickr8k/Images"):
  !unzip "/content/flickr8k.zip" -d "/content/flickr8k"

In [54]:
print("Images:", len(os.listdir("/content/flickr8k/Images")))

Images: 8091


In [55]:
# Copying 30k to Colabs local SSD
if not os.path.exists("/content/flickr30k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr30k.zip" /content/

In [56]:
if not os.path.isdir("/content/flickr30k/Images"):
  !unzip "/content/flickr30k.zip" -d "/content/flickr30k"

In [57]:
print("Images:", len(os.listdir("/content/flickr30k/Images")))

Images: 31811


In [58]:
DATASETS = {
    "flickr8k": {
        "ROOT": "/content/flickr8k",
        "IMAGE_DIR": "/content/flickr8k/Images",
        "CAPTION_FILE": "/content/flickr8k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/flickr8k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/vocab.pkl"
    },
    "flickr30k": {
        "ROOT": "/content/flickr30k",
        "IMAGE_DIR": "/content/flickr30k/Images",
        "CAPTION_FILE": "/content/flickr30k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/flickr30k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/vocab.pkl"
    }
}

In [59]:
def image_caption_map(train_df,val_df,test_df):
  train_caption_map = defaultdict(list)
  val_caption_map = defaultdict(list)
  test_caption_map = defaultdict(list)

  bad_img = "861608773_bdafd5c996.jpg"

  train_caption_map.pop(bad_img, None)
  val_caption_map.pop(bad_img, None)
  test_caption_map.pop(bad_img, None)

  for _, row in train_df.iterrows():
      train_caption_map[row["image"]].append(row["caption"])

  for _, row in val_df.iterrows():
      val_caption_map[row["image"]].append(row["caption"])

  for _, row in test_df.iterrows():
      test_caption_map[row["image"]].append(row["caption"])
  return train_caption_map,val_caption_map,test_caption_map



In [60]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [61]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [62]:
# Encoding

def encode_caption(text, vocab):

    tokens = text.split()

    encoded = [vocab["<SOS>"]]

    for token in tokens:
        encoded.append(
            vocab.get(token, vocab["<UNK>"])
        )

    encoded.append(vocab["<EOS>"])

    return encoded

In [63]:
class FlickrRetrievalDataset(Dataset):
    def __init__(self, caption_map, image_dir, vocab, transform=None, random_caption=True):
        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.random_caption = random_caption

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]

        captions = self.caption_map[image_name]

        if self.random_caption:
            caption = random.choice(captions)
        else:
            caption = captions[0]      # fixed caption for eval

        try:
          image = Image.open(
              os.path.join(self.image_dir, image_name)
          ).convert("RGB")

        except Exception:
            return self.__getitem__(
                (idx + 1) % len(self)
            )

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, caption

In [64]:
class FlickrAllCaptionEvalDataset(Dataset):

    def __init__(self, caption_map, image_dir, vocab, transform=None):

        self.samples = []
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform

        for image_name in sorted(caption_map.keys()):

            for caption in caption_map[image_name]:

                self.samples.append(
                    (image_name, caption)
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_name, caption = self.samples[idx]

        image = Image.open(
            os.path.join(self.image_dir, image_name)
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, caption

In [65]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images = []
    captions = []
    lengths = []

    for image, caption in batch:
        images.append(image)
        captions.append(caption)
        lengths.append(len(caption))

    images = torch.stack(images)

    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    lengths = torch.tensor(lengths)

    return images, captions, lengths

In [66]:
def prepare_datasets_dataloaders(train_caption_map,val_caption_map,test_caption_map):

  train_dataset = FlickrRetrievalDataset(train_caption_map,IMAGE_DIR,vocab,train_transform,random_caption=True)
  val_dataset = FlickrRetrievalDataset(val_caption_map,IMAGE_DIR,vocab,image_transform,random_caption=False)
  test_dataset = FlickrRetrievalDataset(test_caption_map,IMAGE_DIR,vocab,image_transform,random_caption=False)
  all_caption_test_dataset = FlickrAllCaptionEvalDataset(test_caption_map,IMAGE_DIR,vocab,image_transform)

  BATCH_SIZE = 256
  train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=8,
    pin_memory=True,
    )
  val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=8,
        pin_memory=True,
    )
  test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=8,
        pin_memory=True,
    )
  all_caption_test_loader = DataLoader(
    all_caption_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=8
    )
  return train_loader,val_loader,test_loader,all_caption_test_loader

ViT(vit_base_patch16_224) Image Encoder

In [67]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

class ViTEncoder(nn.Module):

    def __init__(
        self,
        embed_dim=512,
        freeze_backbone=True,
        dropout=0.15
    ):
        super().__init__()

        self.freeze_backbone = freeze_backbone

        # --------------------------------------------------
        # ViT-B/16
        # Original ImageNet pretrained ViT-B/16
        # --------------------------------------------------

        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )

        # --------------------------------------------------
        # Freeze / unfreeze backbone
        # --------------------------------------------------

        for param in self.vit.parameters():
            param.requires_grad = not freeze_backbone

        # --------------------------------------------------
        # Projection Head
        # 768 -> 1024 -> 512
        # --------------------------------------------------

        self.projection = nn.Sequential(

            nn.Linear(768, 1024),

            nn.LayerNorm(1024),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(1024, embed_dim),

            nn.LayerNorm(embed_dim)
        )

        # --------------------------------------------------
        # Initialization
        # --------------------------------------------------

        for module in self.projection:

            if isinstance(module, nn.Linear):

                nn.init.xavier_uniform_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def forward(self, images):

        # --------------------------------------------------
        # ViT feature extraction
        # --------------------------------------------------

        if self.freeze_backbone:

            with torch.no_grad():

                features = self.vit.forward_features(
                    images
                )

        else:

            features = self.vit.forward_features(
                images
            )

        # --------------------------------------------------
        # ViT output
        #
        # (B, 197, 768)
        #
        # 1 CLS token
        # 196 patch tokens
        # --------------------------------------------------

        cls_token = features[:, 0]

        # --------------------------------------------------
        # Patch tokens
        # --------------------------------------------------

        patch_tokens = features[:, 1:]

        # --------------------------------------------------
        # Mean pooled patch representation
        # --------------------------------------------------

        patch_mean = patch_tokens.mean(
            dim=1
        )

        # --------------------------------------------------
        # Combine CLS + patch information
        # --------------------------------------------------

        features = (
            cls_token +
            patch_mean
        ) / 2.0

        # --------------------------------------------------
        # Projection
        # --------------------------------------------------

        embeddings = self.projection(
            features
        )

        # --------------------------------------------------
        # L2 normalization
        # --------------------------------------------------

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [68]:
# Text Encoder (Transformer Encoder)

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float()
            * (-math.log(10000.0) / embed_dim)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)      # (1, max_len, embed_dim)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TextEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        num_heads=8,
        num_layers=4,
        ff_dim=2048,
        pad_idx=0,
        dropout=0.15
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.position = PositionalEncoding(
            embed_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.projection = nn.Sequential(
            nn.Linear(embed_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(1024, 512),
            nn.LayerNorm(512)
        )

        for module in self.projection:

            if isinstance(module, nn.Linear):

                nn.init.xavier_uniform_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def forward(
        self,
        captions,
        lengths
    ):

        x = self.embedding(captions)

        x = self.position(x)

        device = captions.device

        max_len = captions.size(1)

        lengths = lengths.to(device)

        mask = (
            torch.arange(
                max_len,
                device=device
            )
            .unsqueeze(0)
            .expand(captions.size(0), -1)
            >= lengths.unsqueeze(1)
        )

        x = self.transformer(
            x,
            src_key_padding_mask=mask
        )

        # Masked mean pooling

        valid_mask = (~mask).unsqueeze(-1).float()

        x = (
            x * valid_mask
        ).sum(dim=1) / valid_mask.sum(
            dim=1
        ).clamp(min=1e-8)

        embeddings = self.projection(x)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [69]:
#Joint model
class ViTTransformerRetrieval(nn.Module):

    def __init__(self, image_encoder, text_encoder):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # Learnable CLIP-style temperature
        self.logit_scale = nn.Parameter(
            torch.tensor(np.log(1 / 0.07), dtype=torch.float32)
        )

    def forward(self, images, captions, lengths):

        image_emb = self.image_encoder(images)

        text_emb = self.text_encoder(
            captions,
            lengths
        )

        return image_emb, text_emb

In [70]:
def build_image_text_encoder(vocab,device):
  image_encoder = ViTEncoder(freeze_backbone=True).to(device)

  text_encoder = TextEncoder(
      vocab_size=len(vocab),
      embed_dim=512,
      pad_idx=vocab["<PAD>"]
  ).to(device)

  model = ViTTransformerRetrieval(
      image_encoder=image_encoder,
      text_encoder=text_encoder
  ).to(device)

  return model

In [71]:
# CLIP-Style Contrastive Loss

def clip_contrastive_loss(image_emb, text_emb, logit_scale):

    # CLIP-style learnable temperature
    logit_scale = logit_scale.exp().clamp(max=100)

    # Similarity Matrix
    logits = torch.matmul(image_emb, text_emb.T) * logit_scale

    # Ground truth labels
    targets = torch.arange(
        image_emb.size(0),
        device=image_emb.device
    )

    # Image → Text
    loss_i2t = F.cross_entropy(logits, targets)

    # Text → Image
    loss_t2i = F.cross_entropy(logits.T, targets)

    # Bidirectional InfoNCE
    loss = (loss_i2t + loss_t2i) / 2

    return loss, logits

In [72]:
def define_optimizer(model):

    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.image_encoder.projection.parameters(),
                "lr": 3e-4
            },
            {
                "params": model.text_encoder.parameters(),
                "lr": 3e-4
            },
            {
                "params": [model.logit_scale],
                "lr": 1e-4
            }
        ],
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=50
    )

    return optimizer, scheduler

In [73]:
# Add Validation Loop
def evaluate(model, dataloader, device):

    model.eval()

    total_loss = 0.0

    with torch.inference_mode():

        for images, captions, lengths in dataloader:

            images = images.to(
                device,
                non_blocking=True
            )

            captions = captions.to(
                device,
                non_blocking=True
            )
            lengths = lengths.to(
                device,
                non_blocking=True
            )

            image_emb, text_emb = model(
                images,
                captions,
                lengths
            )

            loss, _ = clip_contrastive_loss(
                image_emb,
                text_emb,
                model.logit_scale
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [74]:
def model_training(model,train_loader,val_loader,optimizer,scheduler,dataset_name):
  import time
  import torch

  # Training
  NUM_EPOCHS = 50
  best_val_loss = float("inf")
  patience = 8
  epochs_without_improvement = 0

  SAVE_DIR = f"/content/drive/MyDrive/MMRetrieval/R5/{dataset_name}"
  os.makedirs(SAVE_DIR, exist_ok=True)

  BEST_MODEL_PATH = os.path.join(
      SAVE_DIR,
      "best_retrieval_R5_model.pth"
  )

  for epoch in range(NUM_EPOCHS):

      print(f"\n================ Epoch {epoch+1}/{NUM_EPOCHS} ================")

      model.train()
      running_loss = 0.0

      epoch_start = time.time()

      for batch_idx, (images, captions, lengths) in enumerate(train_loader):

          batch_start = time.time()

          images = images.to(device, non_blocking=True)
          captions = captions.to(device, non_blocking=True)
          lengths = lengths.to(
                device,
                non_blocking=True
            )

          optimizer.zero_grad(set_to_none=True)

          image_emb, text_emb = model(
              images,
              captions,
              lengths
          )

          loss, _ = clip_contrastive_loss(
              image_emb,
              text_emb,
              model.logit_scale
          )

          loss.backward()

          torch.nn.utils.clip_grad_norm_(
              model.parameters(),
              max_norm=1.0
          )

          optimizer.step()

          running_loss += loss.item()

          batch_time = time.time() - batch_start

          print(
              f"Batch {batch_idx+1:02d}/{len(train_loader)} | "
              f"Loss: {loss.item():.4f} | "
              f"Time: {batch_time:.2f}s"
          )

      train_time = time.time() - epoch_start

      train_loss = running_loss / len(train_loader)

      # ---------------- Validation ----------------
      val_start = time.time()

      val_loss = evaluate(
          model,
          val_loader,
          device
      )

      val_time = time.time() - val_start

      scheduler.step()

      print("\n---------------- Summary ----------------")
      print(f"Train Loss     : {train_loss:.4f}")
      print(f"Validation Loss: {val_loss:.4f}")
      print(f"Training Time  : {train_time:.2f} sec")
      print(f"Validation Time: {val_time:.2f} sec")
      print(
          f"GPU Memory Used: "
          f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
      )

      # Save best model
      if val_loss < best_val_loss:

          best_val_loss = val_loss
          epochs_without_improvement = 0

          torch.save(
              {
                  "model_state_dict": model.state_dict(),
                  "optimizer_state_dict": optimizer.state_dict(),
                  "epoch": epoch,
                  "scheduler_state_dict": scheduler.state_dict(),
                  "val_loss": val_loss
              },
              BEST_MODEL_PATH
          )
          print(f"✓ Best model saved(Val Loss: {val_loss:.4f})")
      else:
        epochs_without_improvement += 1
        print(f"No improvement for "f"{epochs_without_improvement}/{patience} epochs")

      if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered!")
        break

  FINAL_MODEL_PATH = BEST_MODEL_PATH
  checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
  model.load_state_dict(checkpoint["model_state_dict"])

  return model,FINAL_MODEL_PATH

In [75]:
def extract_embeddings(model, dataloader, device):
    model.eval()

    image_embeddings = []
    text_embeddings = []

    with torch.inference_mode():
        for images, captions, lengths in dataloader:
            images = images.to(device, non_blocking=True)
            captions = captions.to(device, non_blocking=True)
            lengths = lengths.to(
                device,
                non_blocking=True
            )

            img_emb, txt_emb = model(images, captions, lengths)

            image_embeddings.append(img_emb.cpu())
            text_embeddings.append(txt_emb.cpu())

    image_embeddings = torch.cat(image_embeddings, dim=0)
    text_embeddings = torch.cat(text_embeddings, dim=0)

    return image_embeddings, text_embeddings

In [76]:
# image -> text
def image_to_text_recall(similarity, k):

    correct = 0

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        topk = similarity[img_idx].topk(k).indices.tolist()

        if any(idx in gt_caps for idx in topk):
            correct += 1

    return correct / similarity.shape[0]


In [77]:
def text_to_image_recall(similarity, k):

    similarity_t = similarity.T

    correct = 0

    for cap_idx in range(similarity_t.shape[0]):

        gt_img = cap_idx // 5

        topk = similarity_t[cap_idx]\
            .topk(k)\
            .indices\
            .tolist()

        if gt_img in topk:
            correct += 1

    return correct / similarity_t.shape[0]


In [78]:
def image_to_text_mrr(similarity):

    reciprocal_ranks = []

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        sorted_idx = torch.argsort(
            similarity[img_idx],
            descending=True
        )

        best_rank = float("inf")

        for cap in gt_caps:

            rank = (
                (sorted_idx == cap)
                .nonzero(as_tuple=True)[0]
                .item()
            ) + 1

            best_rank = min(best_rank, rank)

        reciprocal_ranks.append(
            1.0 / best_rank
        )

    return np.mean(reciprocal_ranks)

In [79]:
def text_to_image_mrr(similarity):

    similarity_t2i = similarity.T

    reciprocal_ranks = []

    for cap_idx in range(similarity_t2i.shape[0]):

        gt_image = cap_idx // 5

        sorted_idx = torch.argsort(
            similarity_t2i[cap_idx],
            descending=True
        )

        rank = (
            (sorted_idx == gt_image)
            .nonzero(as_tuple=True)[0]
            .item()
        ) + 1

        reciprocal_ranks.append(
            1.0 / rank
        )

    return np.mean(reciprocal_ranks)

In [80]:
def model_testing(model,FINAL_MODEL_PATH,all_caption_test_loader):


  model.eval()
  with torch.inference_mode():
    image_embs, text_embs = extract_embeddings(
      model,
      all_caption_test_loader,
      device
    )
  print(image_embs.shape)
  print(text_embs.shape)
  # Similarity Matrix
  unique_image_embs = image_embs[::5]

  similarity = unique_image_embs @ text_embs.T

  print("Image embeddings:", unique_image_embs.shape)
  print("Text embeddings:", text_embs.shape)
  print("Similarity:", similarity.shape)
  results = pd.DataFrame({
      "Metric": [
          "Recall@1",
          "Recall@5",
          "Recall@10",
          "MRR"
      ],
      "Image→Text": [
          image_to_text_recall(similarity, 1),
          image_to_text_recall(similarity, 5),
          image_to_text_recall(similarity, 10),
          image_to_text_mrr(similarity)
      ],
      "Text→Image": [
          text_to_image_recall(similarity, 1),
          text_to_image_recall(similarity, 5),
          text_to_image_recall(similarity, 10),
          text_to_image_mrr(similarity)
      ]
  })

  results["Image→Text"] = results["Image→Text"].round(6)
  results["Text→Image"] = results["Text→Image"].round(6)

  display(results)
  return

In [81]:
import gc
def reset_resourses():
  gc.collect()
  torch.cuda.empty_cache()
  print("\nMemory after cleanup")
  print("Allocated:",torch.cuda.memory_allocated()/1024**3)
  print("Reserved:",torch.cuda.memory_reserved()/1024**3)

In [82]:
for dataset_name, cfg in DATASETS.items():
    torch.cuda.reset_peak_memory_stats()
    print("\n************************")
    print(f"\nProcessing - {dataset_name}")
    print("\n************************\n")
    ROOT = cfg["ROOT"]
    IMAGE_DIR = cfg["IMAGE_DIR"]
    CAPTION_FILE = cfg["CAPTION_FILE"]
    with open(cfg["flickr_split"],"rb") as f:
      split = pickle.load(f)

    with open(cfg["vocab"],"rb") as g:
      vocab = pickle.load(g)

    model = build_image_text_encoder(vocab, device)

   # print("MODEL\n",model)

    optimizer,scheduler = define_optimizer(model)

    df = pd.read_csv(CAPTION_FILE)
    train_imgs = split["train"]
    val_imgs = split["val"]
    test_imgs = split["test"]

    print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

    train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
    val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)
    test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

    train_df = train_df.dropna(subset=["caption"]).reset_index(drop=True)
    val_df = val_df.dropna(subset=["caption"]).reset_index(drop=True)
    test_df = test_df.dropna(subset=["caption"]).reset_index(drop=True)

    train_caption_map,val_caption_map,test_caption_map = image_caption_map(train_df,val_df,test_df)

    train_loader,val_loader,test_loader,all_caption_test_loader = prepare_datasets_dataloaders(train_caption_map,val_caption_map,test_caption_map)

    best_model,FINAL_MODEL_PATH  = model_training(model,train_loader,val_loader,optimizer,scheduler,dataset_name)

    model_testing(best_model,FINAL_MODEL_PATH,all_caption_test_loader)
    del model
    del optimizer
    del scheduler

    del train_loader
    del val_loader
    del test_loader
    del all_caption_test_loader

    del train_df
    del val_df
    del test_df

    del train_caption_map
    del val_caption_map
    del test_caption_map

    del split
    del vocab
    del df

    reset_resourses()


************************

Processing - flickr8k

************************



/tmp/ipykernel_6999/3535708442.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Train: 6068 | Val: 1011 | Test: 1012

================ Epoch 1/50 ================
Batch 01/24 | Loss: 5.6258 | Time: 0.60s
Batch 02/24 | Loss: 5.5700 | Time: 0.57s
Batch 03/24 | Loss: 5.5437 | Time: 0.56s
Batch 04/24 | Loss: 5.5458 | Time: 0.56s
Batch 05/24 | Loss: 5.4863 | Time: 0.56s
Batch 06/24 | Loss: 5.4605 | Time: 0.55s
Batch 07/24 | Loss: 5.3363 | Time: 0.56s
Batch 08/24 | Loss: 5.2311 | Time: 0.56s
Batch 09/24 | Loss: 5.1194 | Time: 0.56s
Batch 10/24 | Loss: 5.1545 | Time: 0.56s
Batch 11/24 | Loss: 5.0394 | Time: 0.56s
Batch 12/24 | Loss: 5.0537 | Time: 0.56s
Batch 13/24 | Loss: 4.9555 | Time: 0.55s
Batch 14/24 | Loss: 5.0006 | Time: 0.56s
Batch 15/24 | Loss: 4.9249 | Time: 0.56s
Batch 16/24 | Loss: 4.7520 | Time: 0.56s
Batch 17/24 | Loss: 4.7560 | Time: 0.56s
Batch 18/24 | Loss: 4.7642 | Time: 0.57s
Batch 19/24 | Loss: 4.6698 | Time: 0.57s
Batch 20/24 | Loss: 4.6709 | Time: 0.56s
Batch 21/24 | Loss: 4.5999 | Time: 0.56s
Batch 22/24 | Loss: 4.4655 | Time: 0.56s
Batch 23/24 | L

,Metric,Image→Text,Text→Image
0,Recall@1,0.314229,0.228458
1,Recall@5,0.626482,0.531818
2,Recall@10,0.748024,0.672530
3,MRR,0.456687,0.371273



Memory after cleanup
Allocated: 0.47182273864746094
Reserved: 2.171875

************************

Processing - flickr30k

************************



/tmp/ipykernel_6999/3535708442.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Streaming output truncated to the last 5000 lines.
Validation Time: 10.89 sec
GPU Memory Used: 1.10 GB
✓ Best model saved(Val Loss: 2.9447)

================ Epoch 3/50 ================
Batch 01/94 | Loss: 2.9935 | Time: 0.60s
Batch 02/94 | Loss: 2.9388 | Time: 0.58s
Batch 03/94 | Loss: 3.0149 | Time: 0.58s
Batch 04/94 | Loss: 3.0593 | Time: 0.58s
Batch 05/94 | Loss: 3.0572 | Time: 0.60s
Batch 06/94 | Loss: 3.3051 | Time: 0.57s
Batch 07/94 | Loss: 3.0444 | Time: 0.57s
Batch 08/94 | Loss: 3.1230 | Time: 0.56s
Batch 09/94 | Loss: 3.2037 | Time: 0.57s
Batch 10/94 | Loss: 3.1700 | Time: 0.57s
Batch 11/94 | Loss: 2.9460 | Time: 0.58s
Batch 12/94 | Loss: 3.0794 | Time: 0.59s
Batch 13/94 | Loss: 2.9594 | Time: 0.60s
Batch 14/94 | Loss: 2.7342 | Time: 0.58s
Batch 15/94 | Loss: 3.2104 | Time: 0.57s
Batch 16/94 | Loss: 3.1639 | Time: 0.57s
Batch 17/94 | Loss: 2.9597 | Time: 0.57s
Batch 18/94 | Loss: 2.9053 | Time: 0.57s
Batch 19/94 | Loss: 2.9819 | Time: 0.59s
Batch 20/94 | Loss: 2.9966 | Time: 

,Metric,Image→Text,Text→Image
0,Recall@1,0.261515,0.177397
1,Recall@5,0.537126,0.423861
2,Recall@10,0.662472,0.544375
3,MRR,0.392322,0.295933



Memory after cleanup
Allocated: 0.4864997863769531
Reserved: 2.11328125
